# Podstawy sztucznej inteligencji — Laboratorium 3
## Wiedza, wnioskowanie i grafy wiedzy

Na wykładzie 4 zobaczyliśmy, jak zapisać wiedzę w maszynie i jak wyciągać z niej wnioski. Dziś **nie piszemy algorytmów od zera**. Wszystkie funkcje są gotowe i opisane komentarzami. Waszym zadaniem jest:

- **czytać** gotowy kod i rozumieć, co robi,
- **przewidywać** wynik, zanim uruchomicie komórkę,
- **zmieniać wiedzę**: dopisywać fakty, reguły, trójki i liczby,
- **odpowiadać** krótko na pytania w komórkach tekstowych.

| Część | Temat | Na zajęciach | Do domu |
|---|---|---|---|
| 1 | Baza wiedzy i wnioskowanie w przód | zadania 1.1–1.3 | |
| 2 | Wnioskowanie wstecz | zadania 2.1–2.2 | 2.3, ★ 2.4 |
| 3 | Wielu studentów, świat zamknięty i otwarty | | cała część |
| 4 | Ontologia jako trójki i reasoner | zadania 4.1–4.2 | 4.3–4.5, ★ 4.6 |
| 5 | Wikidata i SPARQL | | cała część |
| | Podsumowanie | | pytania na końcu |

**Jak pracujemy**
- Najpierw zapisz kopię: *Plik → Zapisz kopię na Dysku*.
- Uruchamiaj komórki po kolei (Shift+Enter). Każda część zaczyna się od komórki **„Gotowe narzędzia”**: uruchom ją i przeczytaj komentarze.
- Miejsca do zmiany są oznaczone `# TODO`. Zwykle to jedna linia: fakt, reguła albo liczba.
- Po zadaniu często jest komórka z testem. Gdy wszystko działa, zobaczysz ✅.
- Zadania z ★ są dla chętnych.
- Przykład przewodni jest ten sam co na wykładzie: uczelnia, Ania, PSI, Kowalski.

In [ ]:
# Uruchom tę komórkę na początku: importy bibliotek
import random              # losowa kolejność reguł (zadanie 2.3)
import pandas as pd        # wyniki w postaci tabel

def ok(nazwa):
    # Wypisuje komunikat, gdy test przeszedł.
    print(f"✅ {nazwa}: wszystko działa")

print("Gotowe")

---
# Część 1. Baza wiedzy i wnioskowanie w przód

### Jak zapisujemy wiedzę?

- **Fakt** to napis, np. `"OddałaZadania"`. Wszystkie fakty trzymamy w zbiorze (`set`).
- **Reguła** to trójka `(nazwa, lista przesłanek, wniosek)`. Czytamy: *jeśli wszystkie przesłanki są prawdą, to wniosek jest prawdą*.

Baza wiedzy o Ani z wykładu (slajd 22):

| Reguła | Przesłanki → wniosek |
|---|---|
| R1 | OddałaZadania ∧ ZdałaKolokwium → ZaliczonyLab |
| R2 | ZaliczonyLab → Dopuszczona |
| R3 | ZaliczonyLab → MożeOddaćProjekt |
| R4 | Dopuszczona ∧ ZdałaEgzamin → ZaliczonyPrzedmiot |

In [ ]:
# Fakty o Ani: to, co wiemy na pewno
FAKTY = {"OddałaZadania", "ZdałaKolokwium"}

# Reguły: (nazwa, [przesłanka 1, przesłanka 2, ...], wniosek)
REGULY = [
    ("R1", ["OddałaZadania", "ZdałaKolokwium"], "ZaliczonyLab"),
    ("R2", ["ZaliczonyLab"], "Dopuszczona"),
    ("R3", ["ZaliczonyLab"], "MożeOddaćProjekt"),
    ("R4", ["Dopuszczona", "ZdałaEgzamin"], "ZaliczonyPrzedmiot"),
]

In [ ]:
# ===== Gotowe narzędzia: wnioskowanie w przód =====

def wnioskuj_w_przod(fakty, reguly, pokaz=True):
    '''
    Wnioskowanie w przód (slajd 23).

    Parametry:
        fakty  - zbiór faktów, od których zaczynamy
        reguly - lista reguł (nazwa, przesłanki, wniosek)
        pokaz  - True: wypisuj kolejne kroki, False: licz po cichu

    Zwraca:
        zbiór wszystkich faktów: tych na starcie i wywnioskowanych.
    '''
    fakty = set(fakty)    # robimy kopię, żeby nie zmienić zbioru podanego z zewnątrz
    runda = 0

    while True:           # powtarzamy, aż nic nowego się nie pojawi
        runda += 1
        nowe = set()      # nowe fakty znalezione w tej rundzie

        # Przechodzimy przez wszystkie reguły po kolei
        for nazwa, przeslanki, wniosek in reguly:
            # Reguła "odpala", gdy:
            #  1) wszystkie jej przesłanki są już faktami,
            #  2) jej wniosku jeszcze nie znamy (inaczej nic nowego nie wnosi).
            if all(p in fakty for p in przeslanki) and wniosek not in fakty:
                nowe.add(wniosek)
                if pokaz:
                    print(f"Runda {runda}: {nazwa} odpala → {wniosek}")

        # Punkt stały: w całej rundzie nie pojawił się żaden nowy fakt -> koniec
        if not nowe:
            if pokaz:
                print(f"Runda {runda}: nic nowego, STOP")
            return fakty

        # Dopisujemy nowe fakty do bazy dopiero po rundzie
        # (dlatego w rundzie 2 odpalają naraz R2 i R3, jak na slajdzie 25)
        fakty |= nowe

print("Narzędzia z części 1 gotowe")

### Zadanie 1.1. Przewidź, potem uruchom

Zanim uruchomisz komórkę, zapisz w komórce tekstowej poniżej:
1. Ile rund wykona algorytm?
2. Które fakty zostaną wywnioskowane?
3. Dlaczego R4 nie odpali?

Potem uruchom i porównaj ze slajdami 24–26.

In [ ]:
wynik = wnioskuj_w_przod(FAKTY, REGULY)
print()
print("Wszystkie fakty:", sorted(wynik))
print("Nowe fakty:     ", sorted(wynik - FAKTY))

**Moje przewidywania (1.1):**
1. Algorytm wykona 3 rundy. W trzeciej rundzie nie pojawi się już nowy fakt.
2. Zostaną wywnioskowane: `ZaliczonyLab`, `Dopuszczona` i `MożeOddaćProjekt`.
3. R4 nie odpali, ponieważ w bazie nie ma faktu `ZdałaEgzamin`.


### Zadanie 1.2. TELL: dodaj fakt

Agent oparty na wiedzy ma dwie operacje: **TELL** (dodaj wiedzę) i **ASK** (zapytaj). Wiedza jest oddzielona od programu: zmieniamy dane, a nie kod funkcji.

Ania zdała egzamin. Dopisz ten fakt do zbioru `FAKTY_2` i uruchom wnioskowanie jeszcze raz. W której rundzie odpala R4?

In [ ]:
FAKTY_2 = FAKTY | {
    "ZdałaEgzamin",
}
wnioskuj_w_przod(FAKTY_2, REGULY);


In [ ]:
# Test 1.2
assert "ZaliczonyPrzedmiot" in wnioskuj_w_przod(FAKTY_2, REGULY, pokaz=False), "Ania powinna zaliczyć przedmiot"
ok("Zadanie 1.2")

### Zadanie 1.3. Dopisz własne reguły

Dopisz **co najmniej 3 reguły** tak, żeby z bazy wynikał fakt `"Stypendium"`. Reguły mają tworzyć **łańcuch**: przesłanka jednej reguły to wniosek innej. Przykładowy pomysł: zaliczony przedmiot i średnia powyżej 4,5 → wysoka średnia roczna → stypendium. Jeśli Twoja reguła potrzebuje nowego faktu (np. o średniej), dopisz go do `MOJE_FAKTY`.

Sprawdź, w której rundzie odpala każda Twoja reguła.

In [ ]:
MOJE_FAKTY = FAKTY_2 | {
    "ŚredniaPowyżej4,5",
}

MOJE_REGULY = [
    ("R5", ["ZaliczonyPrzedmiot", "ŚredniaPowyżej4,5"], "WysokaŚredniaRoczna"),
    ("R6", ["WysokaŚredniaRoczna"], "SpełniaKryteriaStypendium"),
    ("R7", ["SpełniaKryteriaStypendium"], "Stypendium"),
]

wnioskuj_w_przod(MOJE_FAKTY, REGULY + MOJE_REGULY);


In [ ]:
# Test 1.3
assert len(MOJE_REGULY) >= 3, "Dopisz co najmniej 3 reguły"
assert "Stypendium" in wnioskuj_w_przod(MOJE_FAKTY, REGULY + MOJE_REGULY, pokaz=False), "Z bazy nie wynika Stypendium"
wnioski = {w for _, _, w in MOJE_REGULY}
assert any(p in wnioski for _, prz, _ in MOJE_REGULY for p in prz), "Reguły mają tworzyć łańcuch"
ok("Zadanie 1.3")

---
# Część 2. Wnioskowanie wstecz

Teraz zaczynamy od **pytania** (slajdy 27–30). Przeczytaj komentarze w funkcji. Zwróć uwagę na trzy rzeczy:
- funkcja wywołuje samą siebie (rekurencja), tak jak DFS z wykładu 2,
- gdy jedna reguła zawiedzie, próbuje następnej (nawrót),
- pamięta, które cele już są „w drodze”, żeby nie wpaść w pętlę.

In [ ]:
# ===== Gotowe narzędzia: wnioskowanie wstecz =====

def wnioskuj_wstecz(cel, fakty, reguly, pokaz=True, _glebokosc=0, _w_drodze=frozenset()):
    '''
    Wnioskowanie wstecz (slajd 27).

    Parametry:
        cel    - fakt, o który pytamy, np. "Dopuszczona"
        fakty  - zbiór znanych faktów
        reguly - lista reguł (nazwa, przesłanki, wniosek)
        pokaz  - True: wypisz drzewo celów z wcięciami

    Zwraca:
        True, jeśli cel da się udowodnić, w przeciwnym razie False.

    Parametry z podkreśleniem (_glebokosc, _w_drodze) ustawia sama funkcja,
    gdy wywołuje się rekurencyjnie. Nie trzeba ich podawać.
    '''
    wciecie = "  " * _glebokosc          # im głębiej w drzewie, tym większe wcięcie
    def log(tekst):
        if pokaz:
            print(wciecie + tekst)

    # Krok 1: cel jest faktem -> udowodniony od razu
    if cel in fakty:
        log(f"{cel}: fakt ✓")
        return True

    # Ochrona przed cyklem: jeśli ten cel już próbujemy udowodnić wyżej w drzewie,
    # to kręcilibyśmy się w kółko (A potrzebuje B, B potrzebuje A, ...)
    if cel in _w_drodze:
        log(f"{cel}: cykl, przerywam ✗")
        return False

    # Krok 2: szukamy reguł, które mają ten cel jako wniosek
    kandydaci = [r for r in reguly if r[2] == cel]
    if not kandydaci:
        log(f"{cel}: brak faktu i reguły ✗")
        return False

    # Krok 3: próbujemy reguł po kolei. Gdy jedna zawiedzie, bierzemy następną (nawrót)
    for nazwa, przeslanki, _ in kandydaci:
        log(f"{cel}? próbuję {nazwa}: potrzebuję {' i '.join(przeslanki)}")
        # Każda przesłanka staje się nowym celem (rekurencja, o jeden poziom głębiej).
        # all() zatrzymuje się na pierwszej przesłance, której nie da się udowodnić.
        if all(wnioskuj_wstecz(p, fakty, reguly, pokaz, _glebokosc + 1, _w_drodze | {cel})
               for p in przeslanki):
            log(f"{cel}: TAK ({nazwa})")
            return True

    # Krok 4: żadna reguła się nie udała
    log(f"{cel}: NIE")
    return False


def wyjasnij(cel, fakty, reguly, _w_drodze=frozenset()):
    '''
    Moduł wyjaśnień (slajd 7): odpowiada na pytanie "dlaczego?".

    Działa jak wnioskuj_wstecz, ale zamiast True/False zwraca
    listę zdań z dowodem albo None, gdy celu nie da się udowodnić.
    '''
    if cel in fakty:
        return [f"{cel}: fakt"]
    if cel in _w_drodze:
        return None
    for nazwa, przeslanki, wniosek in reguly:
        if wniosek != cel:
            continue
        dowod = [f"{cel}, bo {nazwa}: {' i '.join(przeslanki)}"]
        udalo_sie = True
        for p in przeslanki:
            pod = wyjasnij(p, fakty, reguly, _w_drodze | {cel})   # dowód przesłanki
            if pod is None:
                udalo_sie = False
                break
            dowod += pod
        if udalo_sie:
            return dowod
    return None


def porownaj_algorytmy(lancuchy=100, dlugosc=10):
    '''
    Buduje dużą bazę wiedzy i porównuje pracę obu algorytmów (zadanie 2.3).

    Baza: `lancuchy` niezależnych łańcuchów reguł, każdy o długości `dlugosc`:
        L0_0 → L0_1 → ... → L0_10,   L1_0 → L1_1 → ...,   itd.
    Pytamy tylko o koniec jednego łańcucha.

    Zwraca słownik z liczbą prób zastosowania reguły w każdym algorytmie.
    '''
    fakty = {f"L{i}_0" for i in range(lancuchy)}                  # początki łańcuchów
    reguly = [(f"R{i}_{k}", [f"L{i}_{k}"], f"L{i}_{k+1}")
              for i in range(lancuchy) for k in range(dlugosc)]
    random.shuffle(reguly)                                        # reguły w losowej kolejności
    cel = f"L0_{dlugosc}"

    # Liczymy próby: w przód każda runda sprawdza wszystkie reguły.
    proby_przod = 0
    f = set(fakty)
    while True:
        proby_przod += len(reguly)
        nowe = {w for _, p, w in reguly if all(x in f for x in p)} - f
        if not nowe:
            break
        f |= nowe

    # Wstecz: w tej bazie każdy fakt L_k ma dokładnie jedną regułę, która go daje,
    # więc dowód celu używa jednej reguły na każdy krok łańcucha, czyli `dlugosc` reguł.
    # (Tej liczby nie mierzymy w kodzie: wynika z budowy bazy.)
    proby_wstecz = dlugosc if wnioskuj_wstecz(cel, fakty, reguly, pokaz=False) else None

    return {"reguł w bazie": len(reguly),
            "faktów wywnioskowanych w przód": len(f - fakty),
            "próby w przód": proby_przod,
            "próby wstecz": proby_wstecz}

print("Narzędzia z części 2 gotowe")

### Zadanie 2.1. Zadaj pytania

Uruchom komórkę i porównaj wydruk ze slajdami 28–30. Odpowiedz:
1. Których reguł algorytm w ogóle nie sprawdzał, gdy pytaliśmy o `Dopuszczona`?
2. Dlaczego odpowiedź na `ZaliczonyPrzedmiot` to NIE? Czy to znaczy, że Ania oblała egzamin?
3. Dopisz trzecie pytanie: czy Ania może oddać projekt?

In [ ]:
print(wnioskuj_wstecz("Dopuszczona", FAKTY, REGULY))
print()
print(wnioskuj_wstecz("ZaliczonyPrzedmiot", FAKTY, REGULY))
print()
print(wnioskuj_wstecz("MożeOddaćProjekt", FAKTY, REGULY))


**Odpowiedzi (2.1):**
1. Przy pytaniu o `Dopuszczona` algorytm nie sprawdza R3 i R4, ponieważ nie są potrzebne do dowodu.
2. Odpowiedź NIE oznacza tylko, że celu nie można udowodnić z dostępnych faktów. Nie oznacza, że Ania na pewno oblała egzamin.
3. Fakt `MożeOddaćProjekt` da się udowodnić przez R3, R1 i fakty początkowe.


### Zadanie 2.2. Nawrót: druga droga do celu

Bartek oddał zadania, ale nie zdał kolokwium. Na szczęście regulamin mówi: kto ma zwolnienie z kolokwium, też zalicza laboratorium. To reguła **R5** poniżej.

1. Uruchom komórkę. Bartek nie jest dopuszczony.
2. Dopisz Bartkowi fakt o zwolnieniu. Uruchom jeszcze raz i znajdź w wydruku miejsce, w którym algorytm **wraca** po porażce R1 i próbuje R5.

In [ ]:
REGULY_Z_R5 = REGULY + [("R5", ["OddałaZadania", "MaZwolnienieZKolokwium"], "ZaliczonyLab")]

FAKTY_BARTKA = {
    "OddałaZadania",
    "MaZwolnienieZKolokwium",
}

wnioskuj_wstecz("Dopuszczona", FAKTY_BARTKA, REGULY_Z_R5)


In [ ]:
# Test 2.2
assert wnioskuj_wstecz("Dopuszczona", FAKTY_BARTKA, REGULY_Z_R5, pokaz=False), "Bartek powinien być dopuszczony"
assert "ZdałaKolokwium" not in FAKTY_BARTKA, "Bartek nie zdał kolokwium: nie dopisuj tego faktu"
ok("Zadanie 2.2")

Moduł wyjaśnień pokazuje ten sam dowód krótko, jak w systemach ekspertowych. Uruchom:

In [ ]:
for zdanie in wyjasnij("Dopuszczona", FAKTY_BARTKA, REGULY_Z_R5) or ["Nie da się udowodnić. Wróć do zadania 2.2."]:
    print(zdanie)

### Zadanie 2.3. Kto robi mniej pracy? *(do domu)*

Funkcja `porownaj_algorytmy` buduje dużą bazę i liczy, ile razy każdy algorytm próbował zastosować regułę. Uruchom ją dla kilku rozmiarów bazy i uzupełnij tabelę w komórce tekstowej.

In [ ]:
wyniki_porownania = []
for liczba_lancuchow in [10, 100, 1000]:
    wynik = porownaj_algorytmy(lancuchy=liczba_lancuchow, dlugosc=10)
    wyniki_porownania.append((liczba_lancuchow, wynik))
    print(liczba_lancuchow, wynik)


| Łańcuchy | Reguł w bazie | Próby w przód | Próby wstecz |
|---|---:|---:|---:|
| 10 | 100 | 1 100 | 10 |
| 100 | 1 000 | 11 000 | 10 |
| 1000 | 10 000 | 110 000 | 10 |

**Wnioski:**
1. Praca wnioskowania w przód rośnie wraz z całą bazą, natomiast wstecz pozostaje skupione na jednym potrzebnym łańcuchu.
2. Wnioskowanie w przód jest lepsze, gdy chcemy od razu wyprowadzić wszystkie konsekwencje, np. w monitoringu, alarmach i regułach biznesowych.


### ★ Zadanie 2.4. Pułapka: cykl *(do domu)*

Ktoś dopisał dwie reguły, które wskazują na siebie nawzajem. Uruchom komórkę. Znajdź w kodzie `wnioskuj_wstecz` linie, które chronią przed nieskończoną rekurencją. Co by się stało bez nich? (Jeśli chcesz sprawdzić: skopiuj funkcję, usuń ochronę i uruchom. Zobaczysz `RecursionError`.)

Ochronę zapewnia warunek `if cel in _w_drodze`. Bez niego funkcja wywoływałaby się bez końca dla A i B, aż Python zgłosiłby `RecursionError`.


In [ ]:
R_cykl = [("C1", ["B"], "A"),
          ("C2", ["A"], "B")]
wnioskuj_wstecz("A", {"X"}, R_cykl)

---
# Część 3. Wielu studentów. Świat zamknięty i otwarty *(do domu)*

In [ ]:
# ===== Gotowe narzędzia: pytania o wielu studentów =====

# Każdy student ma swój zbiór faktów. Reguły (REGULY) są wspólne dla wszystkich.
STUDENCI = {
    "ania":   {"OddałaZadania", "ZdałaKolokwium"},
    "bartek": {"OddałaZadania"},
    "celina": {"OddałaZadania", "ZdałaKolokwium", "ZdałaEgzamin"},
    "darek":  set(),          # o Darku nic nie wiemy
}

# Fakty, o których wiemy na pewno, że są FAŁSZEM (np. z protokołu)
WIADOMO_FALSZ = {
    "bartek": {"ZdałaKolokwium"},
}


def kto(cel, studenci=STUDENCI, reguly=REGULY):
    '''
    Odpowiednik pytania Prologu ze zmienną, np. ?- dopuszczona(Kto).  (slajd 34)
    Zwraca posortowaną listę studentów, dla których cel da się udowodnić.
    '''
    return sorted(s for s, fakty in studenci.items()
                  if wnioskuj_wstecz(cel, fakty, reguly, pokaz=False))


def na_pewno_falsz(cel, falsz, reguly=REGULY):
    '''
    Sprawdza, czy cel jest NA PEWNO fałszem.
    Tak jest, gdy:
      - cel jest na liście faktów fałszywych, albo
      - są reguły z tym wnioskiem i KAŻDA z nich ma przesłankę, która jest na pewno fałszem.
    Upraszczamy: zakładamy, że nasze reguły to jedyne drogi do wniosku.
    '''
    if cel in falsz:
        return True
    kandydaci = [r for r in reguly if r[2] == cel]
    if not kandydaci:
        return False          # nie ma faktu ani reguły: nie wiemy
    return all(any(na_pewno_falsz(p, falsz, reguly) for p in przeslanki)
               for _, przeslanki, _ in kandydaci)


def zapytaj(student, cel, reguly=REGULY):
    '''
    Odpowiedź w świecie otwartym (slajd 32): "TAK", "NIE" albo "NIE WIADOMO".
    '''
    fakty = STUDENCI[student]
    falsz = WIADOMO_FALSZ.get(student, set())
    if wnioskuj_wstecz(cel, fakty, reguly, pokaz=False):
        return "TAK"
    if na_pewno_falsz(cel, falsz, reguly):
        return "NIE"
    return "NIE WIADOMO"


def tabela(cele=("Dopuszczona", "ZaliczonyPrzedmiot")):
    '''Pokazuje obok siebie odpowiedzi w świecie zamkniętym i otwartym.'''
    wiersze = []
    for s in STUDENCI:
        for c in cele:
            zamkniety = "TAK" if wnioskuj_wstecz(c, STUDENCI[s], REGULY, pokaz=False) else "NIE"
            wiersze.append({"student": s, "pytanie": c,
                            "świat zamknięty": zamkniety, "świat otwarty": zapytaj(s, c)})
    return pd.DataFrame(wiersze)

print("Narzędzia z części 3 gotowe")

### Zadanie 3.1. `?- dopuszczona(Kto).`

1. Przewidź: kto jest dopuszczony, a kto zaliczył przedmiot? Potem uruchom.
2. Dopisz do `STUDENCI` nową studentkę `"ewa"`, która jest **dopuszczona**, ale **nie zaliczyła** przedmiotu.

In [ ]:
STUDENCI["ewa"] = {"Dopuszczona"}

print("?- dopuszczona(Kto).        ", kto("Dopuszczona"))
print("?- zaliczony_przedmiot(Kto).", kto("ZaliczonyPrzedmiot"))


In [ ]:
# Test 3.1
assert "ewa" in kto("Dopuszczona"), "Ewa powinna być dopuszczona"
assert "ewa" not in kto("ZaliczonyPrzedmiot"), "Ewa nie powinna mieć zaliczonego przedmiotu"
ok("Zadanie 3.1")

### Zadanie 3.2. „NIE” czy „NIE WIADOMO”?

Uruchom tabelę. Porównaj kolumny: gdzie świat zamknięty i otwarty dają różne odpowiedzi?

Potem dowiadujemy się z protokołu, że **Darek nie oddał zadań**. Dopisz to do `WIADOMO_FALSZ` i uruchom tabelę jeszcze raz. Jak zmieniły się odpowiedzi dla Darka?

In [ ]:
WIADOMO_FALSZ["darek"] = {"OddałaZadania"}

tabela()


In [ ]:
# Test 3.2
assert zapytaj("darek", "Dopuszczona") == "NIE", "Po dopisaniu faktu z protokołu Darek powinien dostać NIE"
assert zapytaj("ania", "ZaliczonyPrzedmiot") == "NIE WIADOMO"
ok("Zadanie 3.2")

**Pytanie (3.2):** Podaj jeden system, w którym słusznie przyjmujemy świat zamknięty, i jeden, w którym byłoby to błędem.

**Odpowiedź:** Świat zamknięty jest uzasadniony w ewidencji magazynowej, w której brak produktu w kompletnej bazie oznacza brak stanu magazynowego. Byłby błędem w systemie medycznym, ponieważ brak informacji o alergii pacjenta nie oznacza, że pacjent na pewno jej nie ma.


---
# Część 4. Ontologia jako trójki i reasoner

Każdą wiedzę da się zapisać jako trójki **(podmiot, predykat, obiekt)**, jak w RDF (slajd 58). Poniżej mała ontologia uczelni. Część pierwsza to **schemat** (klasy i cechy relacji), część druga to **dane** (konkretne osoby i przedmioty).

In [ ]:
ONTOLOGIA = {
    # hierarchia klas: "jest rodzajem" (is-a)
    ("Student",      "subClassOf", "Osoba"),
    ("Pracownik",    "subClassOf", "Osoba"),
    ("Prowadzący",   "subClassOf", "Pracownik"),
    ("Wykład",       "subClassOf", "Zajęcia"),
    ("Laboratorium", "subClassOf", "Zajęcia"),
    # relacja „prowadzi” łączy Prowadzącego (domain) z Przedmiotem (range)
    ("prowadzi", "domain", "Prowadzący"),
    ("prowadzi", "range",  "Przedmiot"),
    # cechy relacji (slajd 57)
    ("wymaga",        "type",      "TransitiveProperty"),   # przechodnia
    ("prowadzi",      "inverseOf", "prowadzonyPrzez"),      # odwrotna
    ("współpracujeZ", "type",      "SymmetricProperty"),    # symetryczna
}

DANE = {
    ("ania",   "type", "Student"),
    ("bartek", "type", "Student"),
    ("ania",   "zapisanaNa", "psi"),
    ("kowalski",   "prowadzi", "psi"),
    ("nowak",      "prowadzi", "mat_dyskretna"),
    ("wisniewska", "prowadzi", "logika"),
    ("psi",           "wymaga", "mat_dyskretna"),
    ("mat_dyskretna", "wymaga", "logika"),
    ("ania", "współpracujeZ", "bartek"),
}

T = ONTOLOGIA | DANE
print(len(T), "trójek")

In [ ]:
# ===== Gotowe narzędzia: reasoner i zapytania =====

def jeden_krok(T):
    '''
    Jeden krok reasonera: stosuje 7 reguł do wszystkich par trójek
    i zwraca zbiór NOWYCH trójek (których jeszcze nie ma w T).

    Reguły (x, y, z to obiekty; A, B, C to klasy; p, q to relacje):
      1. (A subClassOf B), (B subClassOf C)          -> (A subClassOf C)
      2. (x type A), (A subClassOf B)                -> (x type B)
      3. (p domain C), (x p y)                       -> (x type C)
      4. (p range C),  (x p y)                       -> (y type C)
      5. (p type TransitiveProperty), (x p y), (y p z) -> (x p z)
      6. (p inverseOf q), (x p y)                    -> (y q x)
      7. (p type SymmetricProperty), (x p y)         -> (y p x)
    '''
    nowe = set()
    for (s, p, o) in T:                  # pierwsza trójka
        for (s2, p2, o2) in T:           # druga trójka
            if p == "subClassOf" and p2 == "subClassOf" and o == s2:      # reguła 1
                nowe.add((s, "subClassOf", o2))
            if p == "type" and p2 == "subClassOf" and o == s2:            # reguła 2
                nowe.add((s, "type", o2))
            if p == "domain" and p2 == s:                                 # reguła 3
                nowe.add((s2, "type", o))
            if p == "range" and p2 == s:                                  # reguła 4
                nowe.add((o2, "type", o))
            if p == "inverseOf" and p2 == s:                              # reguła 6
                nowe.add((o2, o, s2))
            if p == "type" and o == "SymmetricProperty" and p2 == s:      # reguła 7
                nowe.add((o2, s, s2))
        # reguła 5: relacja przechodnia, potrzebujemy trzeciej trójki (y p z)
        if (p, "type", "TransitiveProperty") in T:
            for (s3, p3, o3) in T:
                if p3 == p and s3 == o:
                    nowe.add((s, p, o3))
    return nowe - T


def reasoner(T, pokaz=True):
    '''
    Reasoner: powtarza jeden_krok, aż nic nowego się nie pojawi (punkt stały).
    To jest wnioskowanie w przód z części 1, tylko na trójkach.
    Zwraca zbiór wszystkich trójek: podanych i wywnioskowanych.
    '''
    T = set(T)
    krok = 0
    while True:
        krok += 1
        nowe = jeden_krok(T)
        if pokaz:
            print(f"Krok {krok}: {len(nowe)} nowych trójek")
        if not nowe:
            return T
        T |= nowe


def znajdz_sprzecznosci(T):
    '''
    Szuka obiektów, które należą do dwóch klas rozłącznych (slajd 60).
    Dla każdej trójki (A, disjointWith, B) sprawdza, czy jakiś x ma typ A i typ B.
    Zwraca listę (x, A, B). Pusta lista = brak sprzeczności.
    '''
    wynik = []
    for (A, p, B) in T:
        if p == "disjointWith":
            for (x, p2, klasa) in T:
                if p2 == "type" and klasa == A and (x, "type", B) in T:
                    wynik.append((x, A, B))
    return sorted(wynik)


def szukaj(T, s=None, p=None, o=None):
    '''
    Wzorzec trójki z dziurami, jak w SPARQL (slajd 67).
    None oznacza "dowolna wartość" (zmienna).
    Przykład: szukaj(T, p="prowadzi") -> wszystkie trójki z relacją "prowadzi".
    Zwraca posortowaną listę pasujących trójek.
    '''
    return sorted(t for t in T
                  if (s is None or t[0] == s)
                  and (p is None or t[1] == p)
                  and (o is None or t[2] == o))


def pokaz_nowe(T_przed, T_po):
    '''Wypisuje w tabeli tylko te trójki, które dopisał reasoner.'''
    return pd.DataFrame(sorted(T_po - T_przed), columns=["podmiot", "predykat", "obiekt"])

print("Narzędzia z części 4 gotowe")

### Zadanie 4.1. Co wywnioskuje reasoner?

1. Uruchom reasoner i przejrzyj nowe trójki.
2. Przy trzech trójkach poniżej wpisz numer reguły z komentarza w `jeden_krok`, która ją dała. Uwaga: jedna z nich wymaga dwóch reguł po kolei.
   - `(psi, wymaga, logika)` …
   - `(psi, prowadzonyPrzez, kowalski)` …
   - `(kowalski, type, Osoba)` …

In [ ]:
T_pelne = reasoner(T)
pokaz_nowe(T, T_pelne)

**Odpowiedzi:**
- `(psi, wymaga, logika)` powstaje przez regułę 5, czyli przechodniość relacji `wymaga`.
- `(psi, prowadzonyPrzez, kowalski)` powstaje przez regułę 6, czyli relację odwrotną.
- `(kowalski, type, Osoba)` wymaga reguły 3, a następnie dwukrotnego użycia reguły 2: prowadzi daje `Prowadzący`, potem hierarchia klas daje `Pracownik` i `Osoba`.


### Zadanie 4.2. Rozbuduj ontologię

Dopisz do `NOWE` trójki opisujące:
- klasę **Doktorant**, która jest podklasą Studenta,
- doktoranta **zielinski**, który prowadzi przedmiot **lab_psi**,
- przedmiot **lab_psi**, który wymaga **psi**.

Nie dopisuj, że zielinski jest Osobą ani że jest Prowadzącym. Niech reasoner sam to znajdzie.

In [ ]:
NOWE = {
    ("Doktorant", "subClassOf", "Student"),
    ("zielinski", "type", "Doktorant"),
    ("zielinski", "prowadzi", "lab_psi"),
    ("lab_psi", "wymaga", "psi"),
}

T2 = reasoner(T | NOWE)
print(szukaj(T2, s="zielinski", p="type"))
print(szukaj(T2, s="lab_psi", p="wymaga"))


In [ ]:
# Test 4.2
assert ("zielinski", "type", "Osoba") not in NOWE, "Tego ma się domyślić reasoner"
for t in [("zielinski", "type", "Osoba"), ("zielinski", "type", "Prowadzący"),
          ("zielinski", "type", "Student"), ("lab_psi", "wymaga", "logika")]:
    assert t in T2, f"Reasoner nie wywnioskował {t}. Sprawdź swoje trójki."
ok("Zadanie 4.2")

### Zadanie 4.3. Reasoner łapie sprzeczności *(do domu)*

1. Ktoś przy imporcie danych zapisał PSI jednocześnie jako Wykład i Salę (slajd 60). Uruchom pierwszą komórkę. Dlaczego sprzeczność widać dopiero **po** reasonerze?
2. Dopisz w drugiej komórce, że **Student** i **Prowadzący** to klasy rozłączne. Uruchom. Kto jest teraz sprzeczny? Czy to błąd w danych, czy w ontologii?

In [ ]:
T_bledne = T | {
    ("Zajęcia", "disjointWith", "Sala"),   # schemat: zajęcia to nie sala
    ("psi", "type", "Wykład"),             # błąd przy imporcie
    ("psi", "type", "Sala"),               # błąd przy imporcie
}
print("Przed reasonerem:", znajdz_sprzecznosci(T_bledne))
print("Po reasonerze:   ", znajdz_sprzecznosci(reasoner(T_bledne, pokaz=False)))

In [ ]:
ROZLACZNE = {
    ("Student", "disjointWith", "Prowadzący"),
}
print(znajdz_sprzecznosci(reasoner(T | NOWE | ROZLACZNE, pokaz=False)))


**Odpowiedzi:**
1. Sprzeczność dotycząca PSI staje się widoczna dopiero po reasonerze, ponieważ najpierw trzeba wywnioskować, że każdy `Wykład` jest także `Zajęciami`.
2. Po uznaniu klas `Student` i `Prowadzący` za rozłączne sprzeczny jest `zielinski`, ponieważ jest doktorantem, a więc studentem, i jednocześnie prowadzi przedmiot. To błąd zbyt restrykcyjnego założenia w ontologii, nie danych.


### Zadanie 4.4. Pytania do grafu: wzorce trójek *(do domu)*

Funkcja `szukaj` działa jak proste zapytanie SPARQL: `None` to dziura. Uzupełnij wywołania tak, żeby odpowiedzieć na pytania. Korzystaj z `T_pelne`.

1. Jakie typy ma kowalski? (gotowe, przykład)
2. Kto prowadzi jakieś zajęcia? (wszystkie trójki z relacją `prowadzi`)
3. Jakie przedmioty trzeba zaliczyć przed PSI?
4. **Pytanie kompetencyjne** ze slajdu 62: kto prowadzi przedmioty, które ania musi zaliczyć przed PSI? Dwa kroki są gotowe, uzupełnij tylko wzorce.

In [ ]:
print("1.", szukaj(T_pelne, s="kowalski", p="type"))
print("2.", szukaj(T_pelne, p="prowadzi"))
print("3.", szukaj(T_pelne, s="psi", p="wymaga"))

przedmioty = [t[2] for t in szukaj(T_pelne, s="psi", p="wymaga")]
prowadzacy = sorted(t[0] for z in przedmioty for t in szukaj(T_pelne, p="prowadzi", o=z))
print("4.", prowadzacy)


In [ ]:
# Test 4.4
assert sorted(przedmioty) == ["logika", "mat_dyskretna"], "Krok 1: sprawdź wzorzec (psi, wymaga, ?)"
assert prowadzacy == ["nowak", "wisniewska"], "Krok 2: sprawdź wzorzec (?, prowadzi, z)"
ok("Zadanie 4.4")

**Pytanie:** Czy wynik pytania 4 byłby taki sam na `T` przed reasonerem? Dlaczego?

**Odpowiedź:** Nie. W `T` zapisano bezpośrednio tylko, że PSI wymaga matematyki dyskretnej. Dopiero przechodniość relacji `wymaga` dodaje informację, że PSI wymaga również logiki, dlatego przed reasonerem zabrakłoby prowadzącej logikę.


### Zadanie 4.5. Wyjątki: najbliższa informacja wygrywa *(do domu)*

Logika klasyczna nie radzi sobie z pingwinem (slajd 52). Sieć semantyczna szuka cechy w węźle, a jeśli jej tam nie ma, idzie strzałką „jest rodzajem” w górę (slajd 54). Przeczytaj funkcję `cecha`.

1. Uruchom. Sprawdź wynik dla Tweety i Pingu.
2. Dopisz do grafu **strusia Bolka** (Struś to rodzaj Ptaka, strusie nie latają, ale szybko biegają).
3. Dopisz wyjątek na poziomie pojedynczego obiektu: wróbel **Rysio** ma złamane skrzydło i nie lata.

In [ ]:
JEST_RODZAJEM = {          # dziecko -> rodzic (strzałka "jest rodzajem" albo "jest")
    "Tweety":  "Wróbel",
    "Pingu":   "Pingwin",
    "Wróbel":  "Ptak",
    "Pingwin": "Ptak",
    "Ptak":    "Zwierzę",
}
CECHY = {                  # węzeł -> jego własne cechy
    "Zwierzę": {"oddycha": True},
    "Ptak":    {"lata": True, "ma_pióra": True},
    "Pingwin": {"lata": False, "pływa": True},
}

def cecha(wezel, nazwa, pokaz=False):
    '''
    Szuka cechy `nazwa` dla węzła `wezel` w sieci semantycznej.
    Zaczyna od samego węzła i idzie w górę po JEST_RODZAJEM.
    Zwraca pierwszą znalezioną wartość (najbliższa wygrywa) albo None.
    '''
    droga = []
    while wezel is not None:
        droga.append(wezel)
        if nazwa in CECHY.get(wezel, {}):          # cecha zapisana w tym węźle?
            if pokaz:
                print(" → ".join(droga), f": {nazwa} = {CECHY[wezel][nazwa]}")
            return CECHY[wezel][nazwa]
        wezel = JEST_RODZAJEM.get(wezel)           # nie ma: idziemy do rodzica
    if pokaz:
        print(" → ".join(droga), f": brak cechy {nazwa}")
    return None

cecha("Tweety", "lata", pokaz=True)
cecha("Pingu", "lata", pokaz=True)
cecha("Pingu", "oddycha", pokaz=True);

In [ ]:
JEST_RODZAJEM["Struś"] = "Ptak"
JEST_RODZAJEM["Bolek"] = "Struś"
CECHY["Struś"] = {"lata": False, "szybko_biega": True}

JEST_RODZAJEM["Rysio"] = "Wróbel"
CECHY["Rysio"] = {"lata": False}

for kto_ in ["Tweety", "Pingu", "Bolek", "Rysio"]:
    cecha(kto_, "lata", pokaz=True)


In [ ]:
# Test 4.5
assert cecha("Bolek", "lata") is False and cecha("Bolek", "ma_pióra") is True, "Bolek: struś nie lata, ale jako ptak ma pióra"
assert cecha("Rysio", "lata") is False, "Rysio: wyjątek zapisz w CECHY dla samego Rysia"
assert cecha("Tweety", "lata") is True
ok("Zadanie 4.5")

### ★ Zadanie 4.6. Prawdziwy SPARQL: biblioteka `rdflib` *(do domu)*

Te same trójki wczytujemy do biblioteki `rdflib` i pytamy prawdziwym SPARQL-em. Zapytanie odpowiada na pytanie kompetencyjne z zadania 4.4. Porównaj je z dwoma krokami w Pythonie: wspólna zmienna `?przedmiot` łączy dwa wzorce.

Zmień zapytanie tak, żeby pokazywało, **kogo uczy kowalski** (reguła z wykładu: Y prowadzi P, X jest zapisany na P → Y uczy X).

In [ ]:
!pip -q install rdflib

In [ ]:
from rdflib import Graph, Namespace

EX = Namespace("http://example.org/uczelnia/")   # prefiks: każda nazwa staje się adresem URI
g = Graph()
for s, p, o in T_pelne:
    g.add((EX[s], EX[p], EX[o]))                  # nasze trójki -> trójki RDF
print(len(g), "trójek w grafie")

zapytanie = '''
PREFIX ex: <http://example.org/uczelnia/>
SELECT ?kto ?przedmiot WHERE {
    ex:psi ex:wymaga ?przedmiot .      # wzorzec 1
    ?kto ex:prowadzi ?przedmiot .      # wzorzec 2 (ta sama zmienna ?przedmiot)
}
'''
for wiersz in g.query(zapytanie):
    print([str(x).split("/")[-1] for x in wiersz])

zapytanie_uczy = """
PREFIX ex: <http://example.org/uczelnia/>
SELECT DISTINCT ?student WHERE {
    ex:kowalski ex:prowadzi ?przedmiot .
    ?student ex:zapisanaNa ?przedmiot .
}
"""
print("Kowalski uczy:")
for wiersz in g.query(zapytanie_uczy):
    print(str(wiersz[0]).split("/")[-1])

---
# Część 5. Wikidata i SPARQL *(do domu)*

Teraz prawdziwy graf wiedzy: Wikidata, ponad 100 milionów elementów. `Q…` to element (rzecz), `P…` to właściwość (relacja) (slajd 66).

| Identyfikator | Znaczenie |
|---|---|
| `wdt:P31` | jest to (instancja klasy) |
| `wdt:P279` | podklasa |
| `wdt:P17` | państwo |
| `wdt:P131` | leży w jednostce administracyjnej |
| `wdt:P1082` | populacja |
| `wdt:P625` | współrzędne |
| `wdt:P27` | obywatelstwo |
| `wdt:P166` | otrzymana nagroda |
| `wd:Q515` | miasto |
| `wd:Q36` | Polska |
| `wd:Q183` | Niemcy |
| `wd:Q580` | Łódź |
| `wd:Q3918` | uniwersytet |
| `wd:Q7191` | Nagroda Nobla |

Identyfikatory innych rzeczy znajdziesz na [wikidata.org](https://www.wikidata.org) (wyszukiwarka u góry strony).

In [ ]:
# ===== Gotowe narzędzia: Wikidata =====
import requests, urllib.parse

def wikidata(zapytanie):
    '''
    Wysyła zapytanie SPARQL do Wikidata i zwraca wynik jako tabelę pandas.
    Gdy połączenie się nie uda, wypisuje błąd i zwraca pustą tabelę.
    '''
    try:
        r = requests.get("https://query.wikidata.org/sparql",
                         params={"query": zapytanie, "format": "json"},
                         headers={"User-Agent": "PSI-lab3-student/1.0 (zajecia dydaktyczne)"},
                         timeout=60)
        r.raise_for_status()                    # błąd HTTP -> wyjątek
    except Exception as e:
        print("Nie udało się połączyć z Wikidata:", e)
        return pd.DataFrame()
    dane = r.json()
    kolumny = dane["head"]["vars"]              # nazwy zmiennych z SELECT
    # Każdy wiersz wyniku to słownik {zmienna: wartość}; brakujące wartości -> None
    wiersze = [{k: (w[k]["value"] if k in w else None) for k in kolumny}
               for w in dane["results"]["bindings"]]
    return pd.DataFrame(wiersze, columns=kolumny)


def link(zapytanie):
    '''Zwraca link, który otwiera zapytanie w przeglądarce query.wikidata.org (np. żeby zobaczyć mapę).'''
    return "https://query.wikidata.org/#" + urllib.parse.quote(zapytanie)

print("Narzędzia z części 5 gotowe")

### Zadanie 5.1. Zapytanie z wykładu

Uruchom zapytanie ze slajdu 67: polskie miasta powyżej 200 000 mieszkańców. Odpowiedz:
1. Czy jakieś miasto pojawia się kilka razy? Dlaczego?
2. Co robi `wdt:P31/wdt:P279*`? Z czym z części 4 to się kojarzy?

In [ ]:
Q_MIASTA = '''
SELECT ?miasto ?miastoLabel ?ludnosc WHERE {
  ?miasto wdt:P31/wdt:P279* wd:Q515 ;    # jest miastem (albo podklasą miasta)
          wdt:P17 wd:Q36 ;               # leży w Polsce
          wdt:P1082 ?ludnosc .           # ma populację
  FILTER(?ludnosc > 200000)
  SERVICE wikibase:label { bd:serviceParam wikibase:language "pl,en". }
}
ORDER BY DESC(?ludnosc)
'''
wikidata(Q_MIASTA)

**Odpowiedzi:**
1. Miasto może wystąpić kilka razy, gdy Wikidata przechowuje kilka wartości populacji, często z różnymi datami lub źródłami.
2. Ścieżka `wdt:P31/wdt:P279*` wybiera obiekty będące instancją miasta albo dowolnej podklasy miasta. Przypomina domknięcie hierarchii `subClassOf` wykonywane przez reasoner.


### Zadanie 5.2. Zmień zapytanie

Skopiowane zapytanie jest poniżej. Zmień **dwie rzeczy**: państwo na Niemcy i próg na 500 000. Potem otwórz wersję z mapą przez link.

In [ ]:
Q_NIEMCY = '''
SELECT DISTINCT ?miasto ?miastoLabel ?ludnosc WHERE {
  ?miasto wdt:P31/wdt:P279* wd:Q515 ;
          wdt:P17 wd:Q183 ;              # Niemcy
          wdt:P1082 ?ludnosc .
  FILTER(?ludnosc > 500000)
  SERVICE wikibase:label { bd:serviceParam wikibase:language "pl,en". }
}
ORDER BY DESC(?ludnosc)
'''
display(wikidata(Q_NIEMCY))

# Wersja z mapą: linia #defaultView:Map i współrzędne ?wsp. Kliknij link.
Q_MAPA = Q_NIEMCY.replace("SELECT DISTINCT ?miasto ?miastoLabel ?ludnosc",
                          "#defaultView:Map\nSELECT DISTINCT ?miasto ?miastoLabel ?wsp") \
                 .replace("wdt:P1082 ?ludnosc .", "wdt:P1082 ?ludnosc ;\n          wdt:P625 ?wsp .")
print(link(Q_MAPA))

### Zadanie 5.3. Uczelnie w Łodzi i polscy nobliści

W obu zapytaniach brakuje identyfikatorów. Uzupełnij `Q____` i `P____` korzystając z tabeli na początku części 5.

Potem odpowiedz: czy na liście noblistów jest Maria Skłodowska-Curie? Jeśli nie, to dlaczego? Co to mówi o świecie otwartym?

In [ ]:
Q_UCZELNIE = '''
SELECT DISTINCT ?uczelnia ?uczelniaLabel WHERE {
  ?uczelnia wdt:P31/wdt:P279* wd:Q3918 ;   # jest uniwersytetem
            wdt:P131 wd:Q580 .              # leży w Łodzi
  SERVICE wikibase:label { bd:serviceParam wikibase:language "pl,en". }
}
ORDER BY ?uczelniaLabel
'''
display(wikidata(Q_UCZELNIE))

Q_NOBLISCI = '''
SELECT DISTINCT ?osoba ?osobaLabel ?nagrodaLabel WHERE {
  ?osoba wdt:P27 wd:Q36 ;                  # obywatelstwo polskie
         wdt:P166 ?nagroda .                # dostał nagrodę
  ?nagroda wdt:P31|wdt:P279 wd:Q7191 .       # Nagroda Nobla
  SERVICE wikibase:label { bd:serviceParam wikibase:language "pl,en". }
}
ORDER BY ?osobaLabel
'''
display(wikidata(Q_NOBLISCI))

**Odpowiedź:** Maria Skłodowska-Curie może nie pojawić się na liście, jeśli w Wikidata nie ma przy niej polskiego obywatelstwa zapisanego dokładnie relacją `P27` albo dane klasyfikują jej obywatelstwo inaczej. W świecie otwartym brak wyniku nie oznacza fałszu, lecz brak danych spełniających podany wzorzec.


---
# Podsumowanie

1. Do systemu pilnującego terminów w USOS wybrałbym wnioskowanie w przód, ponieważ po dodaniu nowego faktu powinien automatycznie wyprowadzić wszystkie konsekwencje i powiadomienia. Do asystenta odpowiadającego „czy mogę podejść do egzaminu?” lepsze jest wnioskowanie wstecz, ponieważ sprawdza tylko reguły potrzebne do odpowiedzi na konkretne pytanie.

2. Reasoner wie, że `zielinski` jest Osobą, ponieważ łączy fakty `zielinski type Doktorant`, `Doktorant subClassOf Student` i `Student subClassOf Osoba`. Inteligencja wynika zarówno z danych opisujących dziedzinę, jak i z ogólnego algorytmu stosującego reguły wnioskowania.

3. Przy 100 000 reguł ręczne utrzymywanie bazy prowadziłoby do sprzeczności, duplikatów i trudnych do wykrycia zależności. Wiedzę można pozyskiwać z danych, tekstów i przykładów z użyciem uczenia maszynowego, ekstrakcji informacji oraz uzgadniania wiedzy wielu ekspertów.

